In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import os
#The directory I used in colab
#os.chdir('/content/drive/My Drive/NLP Project/')
cwd= os.getcwd()
#print ('Current path = ' + cwd)

Mounted at /content/drive
/root


In [ ]:
import re
import csv
import numpy as np


stop_words =['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she', "she'd", "she'll", "she's", 'should', 'shouldn', "shouldn't", "should've", 'so', 'some', 'such', 't', 'than', 'that', "that'll", 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these', 'they', "they'd", "they'll", "they're", "they've", 'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 've', 'very', 'was', 'wasn', "wasn't", 'we', "we'd", "we'll", "we're", 'were', 'weren', "weren't", "we've", 'what', 'when', 'where', 'which', 'while', 'who', 'whom', 'why', 'will', 'with', 'won', "won't", 'wouldn', "wouldn't", 'y', 'you', "you'd", "you'll", 'your', "you're", 'yours', 'yourself', 'yourselves', "you've"]
punctuation=r"!”#%&'()*+,-./:;?@[\]^_`{|}–~"
spampattern=[r"exclus\w*",r"loan\w*",r"offer",r"secur\w*",r"approv\w*",r"won",r"alert",r"computer\w*",r"infect\w*",r"\$\d+(?:\.\d+)?",r"i?phone\w*",r"money",r"crypto\w*",r"free"]
hampattern=[r"meet\w*",r"agenda",r"student",r"report",r"perform\w*",r"sale\w*",r"business*",r"itinerary",r"purchase"]


with open('spam_email_dataset.csv',mode='r',encoding='cp1252') as file:
    csv_read=csv.DictReader(file)
    Data=[]
    for i in csv_read:
        Data.append(i)

FileNotFoundError: [Errno 2] No such file or directory: 'spam_email_dataset.csv'

In [ ]:
def WordFrequency():
    spamsublist=[]
    spambodylist=[]
    hamsublist=[]
    hambodylist=[]
    spamsubdict={}
    spambodydict={}
    hamsubdict={}
    hambodydict={}
    for t in Data:
        txt=t["Subject"]
        clean=wordcleanse(txt)
        if t["Spam Label"]=='1':
            spamsublist.append(clean)
        else:
            hamsublist.append(clean)

    for t in Data:
        txt=t["Body"]
        clean=wordcleanse(txt)
        if t["Spam Label"]=='1':
            spambodylist.append(clean)
        else:
            hambodylist.append(clean)

    for mail in spamsublist:
        email=mail.split()
        for word in email:
            spamsubdict[word]=spamsubdict.get(word,0)+1

    for mail in spambodylist:
        email=mail.split()
        for word in email:
            spambodydict[word]=spambodydict.get(word,0)+1

    for mail in hamsublist:
        email=mail.split()
        for word in email:
            hamsubdict[word]=hamsubdict.get(word,0)+1

    for mail in hambodylist:
        email=mail.split()
        for word in email:
            hambodydict[word]=hambodydict.get(word,0)+1

    #Parses through dataset for the most commonly used words.
    print("Spam email subject words frequency \n",spamsubdict)
    print("\n Spam email body words frequency \n",spambodydict)
    print("\n Ham email subject words frequency \n",hamsubdict)
    print("\n Ham email body words frequency \n",hambodydict)

WordFrequency()

In [ ]:


def wordcleanse(Text):
    not_pun="".join([txt for txt in Text if txt not in punctuation])
    word=not_pun.lower().split()
    Text=" ".join([wrd for wrd in word if wrd not in stop_words])
    return Text

Unsure=0
def classify(text):
    clean=wordcleanse(text)
    score=0
    global Unsure
    detected=[]
    for pattern in spampattern:
        if re.search(pattern,clean):
            detected.append(pattern)
            score+=1
    for pattern in hampattern:
        if re.search(pattern,clean):
            detected.append(pattern)
            score-=1
    if score>0:#Spam
        return detected,'1'
    elif score<0:#Ham
        return detected,'0'
    else:#Inconclusive, guesses Ham
        Unsure+=1
        return detected,'0'

def eval():
    correct=0
    total=0
    matrix=np.zeros((2,2))
    for i in Data:
        Subject=i['Subject']
        Body=i['Body']
        Label=i['Spam Label']
        text=Subject+" "+Body
        detected,prediction=classify(text)
        if prediction==Label:
            correct+=1
        if prediction=='1' and Label=='1':#Matrix Top Left,Guessed spam, is spam
            matrix[0,0]+=1
        elif prediction=='0' and Label=='1':#Matrix Top Right,Guessed ham, is spam
            matrix[0,1]+=1
        elif prediction=='1' and Label=='0':#Matrix Bottom Left,Guessed spam, is ham
            matrix[1,0]+=1
        elif prediction=='0' and Label=='0':#Matrix Bottom Right,Guessed ham, is ham
            matrix[1,1]+=1
        total+=1
    print("Number of correct predictions:",correct,"Total number of emails:",total,"Number of emails the NLP couldn't distinguish:",Unsure)
    accuracy=correct/total
    print("Accuracy:",round(accuracy,2)*100,"%")
    print("Confusion matrix:\n",matrix)
    print("\n")
eval()

def TestCases():
    #To make it easier how the NLP works these are some of test cases
    c=1
    for i in Data:
        #For some exampes of clear cases
        if c<9:
            Subject=i['Subject']
            Body=i['Body']
            Label=i['Spam Label']
            text=Subject+" "+Body
            detected,prediction=classify(text)
            S=0
            H=0
            for w in detected:
                if w in spampattern:
                    S+=1
                elif w in hampattern:
                    H+=1
            print("Test case email",c,"Prediction:","Spam" if prediction=='1' else "Ham","Actuality:","Spam" if Label=='1' else "Ham")
            print("Pattern's detected in email:",detected,"Spam Score:",S,"Ham Score:",H)
            print("This is an example of a clear case that the program could correctly differentiate")
            print("-----------------")
            print(text)
            print("----------------------")
        #For a Spam email that it couldn't differentiate
        elif c==76:
            Subject=i['Subject']
            Body=i['Body']
            Label=i['Spam Label']
            text=Subject+" "+Body
            detected,prediction=classify(text)
            S=0
            H=0
            for w in detected:
                if w in spampattern:
                    S+=1
                elif w in hampattern:
                    H+=1

            print("Test case email",c,"Prediction:","Spam" if prediction=='1' else "Ham","Actuality:","Spam" if Label=='1' else "Ham")
            print("Pattern's detected in email:",detected,"Spam Score:",S,"Ham Score:",H)
            print("This is an example of a spam email that got through that the program couldn't differntiate as it didn't contain any pattern")
            print("-----------------")
            print(text)
            print("----------------------")
        #For a Ham email it couldn't differentiate
        elif c==20:
            Subject=i['Subject']
            Body=i['Body']
            Label=i['Spam Label']
            text=Subject+" "+Body
            detected,prediction=classify(text)
            S=0
            H=0
            for w in detected:
                if w in spampattern:
                    S+=1
                elif w in hampattern:
                    H+=1
            print("Test case email",c,"Prediction:","Spam" if prediction=='1' else "Ham","Actuality:","Spam" if Label=='1' else "Ham")
            print("Pattern's detected in email:",detected,"Spam Score:",S,"Ham Score:",H)
            print("This is an example of a legitamate email that the program couldn't differente as it didn't contain any pattern.")
            print("-----------------")
            print(text)
            print("----------------------")
        elif c==80:
            break
        c+=1
TestCases()
